In [ ]:
%pip install nltk rouge-score
%pip install bert-score
%pip install sentencepiece

# Install TensorFlow for macOS (uncomment the appropriate line for your Mac)

# For Apple Silicon Macs (M1/M2/M3)
%pip install tensorflow-macos tensorflow-metal

# For Intel-based Macs, or fallback option
# %pip install tensorflow==2.10.0

# If the above doesn't work, try a lower version known to work on macOS
# %pip install tensorflow==2.8.0

In [ ]:
# Check if BLEURT-20 directory already exists before downloading
import os

if not os.path.exists('BLEURT-20'):
    print("BLEURT-20 directory not found. Downloading model...")
    # Use curl (Mac OS)
    !curl -L https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip -o BLEURT-20.zip
    !unzip -o BLEURT-20.zip
    print("BLEURT-20 model downloaded and extracted.")
else:
    print("BLEURT-20 directory already exists. Skipping download.")

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download('wordnet')
nltk.download('punkt')

def compute_bleu(reference, candidate):
    """
    Compute BLEU score for BLEU-1, BLEU-2, BLEU-3, and BLEU-4 between reference and candidate.
    Uses a smoothing function for short sentences.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    smoothie = SmoothingFunction().method1  # Smoothing for short sequences
    
    bleu_scores = {}
    for n in range(1, 5):
        weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
        bleu_scores[f"BLEU-{n}"] = sentence_bleu(reference_tokens, candidate_tokens, weights=weights, smoothing_function=smoothie)
    
    return bleu_scores

def compute_rouge(reference, candidate):
    """
    Compute ROUGE-L, ROUGE-1, and ROUGE-2 scores.
    Returns the F1 scores.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return {k: v.fmeasure for k, v in scores.items()}

def compute_meteor(reference, candidate):
    """
    Compute METEOR score.
    """
    reference_tokens = [reference.split()]
    candidate_tokens = candidate.split()
    return meteor_score(reference_tokens, candidate_tokens)

import bert_score.score
def compute_bertscore(reference, candidate, lang='en', model_type='bert-base-uncased'):
    P, R, F1 = bert_score.score([candidate], [reference], lang=lang, model_type=model_type, verbose=False)

    return {
        'precision': P[0].item(),
        'recall': R[0].item(),
        'f1': F1[0].item()
    }

# BLEURT score calculation function
try:
    import bleurt.score
    
    def compute_bleurt(references, candidates, checkpoint_path="BLEURT-20"):
        scorer = bleurt.score.BleurtScorer(checkpoint_path)
        scores = scorer.score(references=references, candidates=candidates)
        return scores
    
    print("BLEURT import successful!")
except ImportError as e:
    print(f"Warning: BLEURT import failed: {e}")
    print("Will use placeholder function for BLEURT")
    
    def compute_bleurt(references, candidates, checkpoint_path="BLEURT-20"):
        print("BLEURT unavailable, returning zero scores")
        return [0.0] * len(references)

In [ ]:
# Check if TensorFlow is available and print version
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

try:
    import tensorflow as tf
    print(f"TensorFlow version: {tf.__version__}")
    print(f"TensorFlow location: {tf.__file__}")
    print("TensorFlow is successfully installed and imported!")
    tf_available = True
except ImportError as e:
    print(f"ImportError: {e}")
    print("TensorFlow could not be imported. BLEURT scores will be set to zero.")
    tf_available = False

In [ ]:
# BLEURT score calculation function - improved detection logic
global_use_bleurt = False  # Will be set to True if BLEURT is successfully imported

try:
    # First check if TensorFlow is available
    import tensorflow as tf
    print(f"Using TensorFlow {tf.__version__}")
    
    # Then try to import BLEURT
    import bleurt.score
    
    def compute_bleurt(references, candidates, checkpoint_path="BLEURT-20"):
        """Compute BLEURT scores between references and candidates."""
        print(f"Initializing BLEURT scorer with checkpoint: {checkpoint_path}")
        scorer = bleurt.score.BleurtScorer(checkpoint_path)
        print(f"Computing BLEURT scores for {len(references)} examples...")
        scores = scorer.score(references=references, candidates=candidates)
        print(f"Successfully computed {len(scores)} BLEURT scores")
        return scores
    
    print("BLEURT import successful! Will compute BLEURT scores.")
    global_use_bleurt = True
    
except ImportError as e:
    print(f"Warning: Cannot use BLEURT: {e}")
    print("If you need BLEURT scores, make sure both TensorFlow and BLEURT are installed:")
    print("  pip install tensorflow-macos")
    print("  pip install git+https://github.com/google-research/bleurt.git")
    
    def compute_bleurt(references, candidates, checkpoint_path="BLEURT-20"):
        print("BLEURT unavailable, returning zero scores")
        return [0.0] * len(references)

In [ ]:
import pandas as pd
import os

results_folder = "results"
result_filename = "results/ablation/pororo_ablation_visual_language_critic_gpt_4o_mini.csv"
evaluation_results_folder = "results/metrics"

# Change this column name based on the CSV file you're processing:
# - For language model: "pure_language_answer"
# - For language+critic: "language_critic_answer"  
# - For visual+language: "visual_language_answer"
# - For visual+language+critic: "visual_language_critic_answer"
answer_column_name = "visual_language_critic_answer"

os.makedirs(evaluation_results_folder, exist_ok=True)

print(f"Processing file: {result_filename}")
result = pd.read_csv(result_filename, dtype={'gif_num': 'Int64', 'row_num': 'Int64'})

# Convert qid to string to remove .0
if 'qid' in result.columns:
    result['qid'] = result['qid'].astype(str).str.replace('.0', '', regex=False)

os.makedirs(evaluation_results_folder, exist_ok=True)

for i, row in result.iterrows():
    if i == len(result)-1: # avoid the last row with total and average values
        continue

    reference_answer = row["correct_answer"].lower()
    # Use the configured answer column name to extract generated answer
    generated_answer = row[answer_column_name].lower()

    bleus = compute_bleu(reference_answer, generated_answer)
    rouges = compute_rouge(reference_answer, generated_answer)
    meteor = compute_meteor(reference_answer, generated_answer)
    berts = compute_bertscore(reference_answer, generated_answer)
    
    for n in range(1, 5):
        result.at[i, f"BLEU-{n}"] = bleus[f"BLEU-{n}"]
    
    for rouge_type, score_value in rouges.items():
        result.at[i, f"{rouge_type.upper()}"] = score_value
    
    result.at[i, "METEOR"] = meteor
    result.at[i, "BERTScore_Precision"] = berts["precision"]
    result.at[i, "BERTScore_Recall"] = berts["recall"]
    result.at[i, "BERTScore_F1"] = berts["f1"]

# Try to calculate BLEURT scores if TensorFlow and BLEURT are available
if global_use_bleurt:
    try:
        print("Calculating BLEURT scores...")
        reference_answers = list(result["correct_answer"])[:-1]
        # Use the configured answer column name to extract generated answers for BLEURT
        generated_answers = list(result[answer_column_name])[:-1]
        bleurts = compute_bleurt(reference_answers, generated_answers)
        result["BLEURT"] = bleurts + [""]
        print("Successfully calculated BLEURT scores")
    except Exception as e:
        print(f"BLEURT calculation failed: {e}")
        print("Skipping BLEURT score calculation")
        # Add zeros for BLEURT scores as fallback
        result["BLEURT"] = [0.0] * (len(result)-1) + [""]
else:
    print("Skipping BLEURT calculation because TensorFlow or BLEURT module is not available")
    # Add zeros for BLEURT scores
    result["BLEURT"] = [0.0] * (len(result)-1) + [""]

# Save results with 'metrics_' prefix
output_filename = "metrics_" + os.path.basename(result_filename)
output_path = os.path.join(evaluation_results_folder, output_filename)

result.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

In [ ]:
result